In [4]:
import os
import pandas as pd
from datasets import load_dataset, DatasetDict, Audio, Dataset, concatenate_datasets
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

In [6]:
def load_local_dataset(PATH):

    def filter_file_exists(df, path_to_audio):
       mask = df['path'].apply(lambda x: os.path.isfile(os.path.join(directory, x)))
       return df[mask]
    
    # Declarer le dataset au format de huggingface (DatasetDict)
    common_voice = DatasetDict()
    
    # chemin absolu vers le datsets 
    directory = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    #chemin vers le fichier audio
    PATH_TO_AUDIO = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/" + PATH + "clips/"

    # Creer les chemins dynamiques
    tsv_path_train = os.path.join(directory, PATH, "train.tsv")
    tsv_path_dev = os.path.join(directory, "..", "datasets", PATH, "dev.tsv")
    tsv_path_test = os.path.join(directory, "..", "datasets", PATH, "test.tsv")

    # Convertir en dataframe pandas
    df_train = pd.read_csv(tsv_path_train, delimiter='\t')
    df_dev = pd.read_csv(tsv_path_dev, delimiter='\t')
    df_test = pd.read_csv(tsv_path_test, delimiter='\t')

    # filtrer le df des lignes avec des noms de fichiers non existants
    df_train = filter_file_exists(df_train, PATH_TO_AUDIO)
    df_dev   = filter_file_exists(df_dev, PATH_TO_AUDIO)
    df_test  = filter_file_exists(df_test, PATH_TO_AUDIO)
    
    # Convertir en dataset huggingface
    common_voice["train"] = Dataset.from_pandas(df_train)
    common_voice["dev"] = Dataset.from_pandas(df_dev)
    common_voice["test"] = Dataset.from_pandas(df_test)

    # Creer la colonne audio pour les trois corpus, elle contient le chemin complet vers l'audio
    common_voice["train"] = common_voice["train"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})
    common_voice["dev"] = common_voice["dev"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})
    common_voice["test"] = common_voice["test"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})

    # On fait le Reechantillonnage des fichiers audio en 16000khz (les audios de common voice sont en 48000khz par defaut) 
    common_voice["train"] = common_voice["train"].cast_column("audio", Audio(sampling_rate=16_000))
    common_voice["dev"] = common_voice["dev"].cast_column("audio", Audio(sampling_rate=16_000))
    common_voice["test"] = common_voice["test"].cast_column("audio", Audio(sampling_rate=16_000))
    
    return common_voice

In [7]:
PATH = 'kurd/'
common_voice = load_local_dataset(PATH)

In [5]:
common_voice['train'].features

{'client_id': Value('string'),
 'path': Value('string'),
 'sentence_id': Value('string'),
 'sentence': Value('string'),
 'sentence_domain': Value('float64'),
 'up_votes': Value('int64'),
 'down_votes': Value('int64'),
 'age': Value('string'),
 'gender': Value('string'),
 'accents': Value('string'),
 'variant': Value('float64'),
 'locale': Value('string'),
 'segment': Value('float64'),
 'audio': Audio(sampling_rate=16000, decode=True, stream_index=None)}

In [6]:
common_voice['train']['audio'].features

Audio(sampling_rate=16000, decode=True, stream_index=None)

In [9]:
common_voice['train']['audio'][350]

In [11]:
audio = common_voice['train']['audio']
audio

RuntimeError: Could not open input file: /info/raid-etu/m1/s2506992/projet-m1-asr/datasets/kurd/clips/common_voice_kmr_34966194.mp3 No such file or directory

In [13]:
audio = common_voice['train']['audio'][350]
audio["array"]

array([ 1.0311446e-13,  8.0208016e-14, -5.7743862e-15, ...,
       -2.8413506e-06, -2.1443498e-06,  2.8284253e-06],
      shape=(55296,), dtype=float32)

In [3]:
def prepare_dataset(batch, feature_extractor, tokenizer):
    # recuperer les colonnes 'audio'
    audio = batch["audio"]

    # on cree une nouvelle colonne 'input_features' qui contiendra les tableaux numpy qui represente le fichier audio 
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # on transforme le text en token numerique
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    
    return batch

In [4]:
processor = WhisperProcessor.from_pretrained("openai/whisper-base",language="turkish", task="transcribe")
feature_extractor = processor.feature_extractor
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")

In [10]:
common_voice = common_voice.map(prepare_dataset, remove_columns=common_voice.column_names["train"], fn_kwargs={"feature_extractor": feature_extractor,  "tokenizer": tokenizer})

In [5]:
df_train.columns

Index(['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain',
       'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant',
       'locale', 'segment'],
      dtype='object')

In [1]:
from transformers import WhisperFeatureExtractor, WhisperProcessor, WhisperForConditionalGeneration, WhisperTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from datasets import load_dataset, DatasetDict, Audio, Dataset, concatenate_datasets
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import pandas as pd
import evaluate
import torch
import os


"""
NB: La methode load_dataset de huggingface ne marche plus pour les nouveaux datasets de common voice, Alors on telecharge le dataset
en local, on le convertis en dataframe pandas, puis on le convertit en dataset huggingface et on fait le reechantillonage des datasets 
"""
def load_local_dataset(PATH):
    # Chemins
    base_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    PATH_TO_AUDIO = os.path.join(base_dir, PATH, "clips")

    # Lire les TSV
    df_train = pd.read_csv(os.path.join(base_dir, PATH, "train.tsv"), delimiter='\t')
    df_dev   = pd.read_csv(os.path.join(base_dir, PATH, "dev.tsv"), delimiter='\t')
    df_test  = pd.read_csv(os.path.join(base_dir, PATH, "test.tsv"), delimiter='\t')

    # Filtrage sans apply
    def filter_files(df):
        filtered_rows = []
        for _, row in df.iterrows():
            audio_file = os.path.join(PATH_TO_AUDIO, row['path'])
            if os.path.isfile(audio_file):
                filtered_rows.append(row)
        return pd.DataFrame(filtered_rows)

    df_train = filter_files(df_train)
    df_dev   = filter_files(df_dev)
    df_test  = filter_files(df_test)

    print("Train examples:", len(df_train))
    print("Dev examples:", len(df_dev))
    print("Test examples:", len(df_test))

    # Convertir en Dataset HuggingFace
    common_voice = DatasetDict({
        "train": Dataset.from_pandas(df_train),
        "dev": Dataset.from_pandas(df_dev),
        "test": Dataset.from_pandas(df_test),
    })

    # Ajouter colonne audio complète
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].map(
            lambda x: {"audio": os.path.join(PATH_TO_AUDIO, x["path"])},
            batched=False
        )

    # Reechantillonnage audio
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].cast_column("audio", Audio(sampling_rate=16_000))

    test_subset = common_voice["test"].select(range(2000))
    common_voice["train"] = concatenate_datasets([common_voice["train"], test_subset])

    return common_voice
"""
cette methode sert a eliminer tout les colonnes non necessaires pour l'apprentissage, on garde une nouvelle colonne: inputfeatures; elle contient un tableau
numpy qui represente les caracteres acoustiques de l'audio, et une colonne label; elle contient la label dans le format Token
"""
def prepare_dataset(batch, feature_extractor, tokenizer):
    # recuperer les colonnes 'audio'
    audio = batch["audio"]

    # on cree une nouvelle colonne 'input_features' qui contiendra les tableaux numpy qui represente le fichier audio 
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

    # on transforme le text en token numerique
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    
    return batch

"""
Un Data Collator sert a preparer le bacth avant de l'envoyer au model, dans notre cas il va servir pour faire 3 choses:
- etape 1: on transforme la colonne 'input features' qui contient les tableaux numpy en tensor pytorch
- etape 2: on unifie la taille des tokens numerique (qui represente les labels) en ajoutant des <pad> qui seront apres remplace par des -100 pour ne pas biaiser 
  les metrics d'evaluation
- etape 3: on supprime le token <s> qui indique le debut de sequence
"""

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # etape 1: 
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # etape 2
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # etape 3
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels

        return batch
        
# Charger la  metric
metric = evaluate.load("wer")

# Defintion des Hyperparametre de l'entrainement
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join("..", "results","whisperBaseTurc"),  # repertoire enregistrement des chekcpoints
    do_train=True,                      
    do_eval=True,  
    per_device_train_batch_size=4,        # petit batch par GPU
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,        # pour batch effectif ~32
    learning_rate=1e-5,
    warmup_steps=50,                      # warmup pour stabiliser
    max_steps=850,                        # 5 epochs complètes
    lr_scheduler_type="cosine",           # scheduler cosinus
    gradient_checkpointing=False,         # économise la mémoire
    fp16=True,                            # demi-précision
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    remove_unused_columns=False
)

def whisper_finetune (PATH):
    
    processor = WhisperProcessor.from_pretrained("openai/whisper-base",language="turkish", task="transcribe")

    feature_extractor = processor.feature_extractor

    tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="turkish", task="transcribe")

    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

    # on recupere le dataset
    common_voice = load_local_dataset(PATH)
    
    # le dataset avec deux colonnes, input_features: spectogramme mel et labels: les tokens numerique des etiquetes
    common_voice = common_voice.map(prepare_dataset, fn_kwargs={"feature_extractor": feature_extractor,  "tokenizer": tokenizer})
    
    # Initialiser le datacollator
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
    )

    """
    elle prends les predictions, elle les compare les predictions et les labels, et retourne la word_error_rate
    """
    def compute_metrics(pred):

        normalizer = BasicTextNormalizer()
    
        # on recupere les predictions
        pred_ids = pred.predictions
    
        # on recupere les labels 
        label_ids = pred.label_ids

        # replace -100 with the pad_token_id
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        # on decode (on convertit les tokens numerique en text) les deux, pred_ids et label_ids
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        pred_str = [normalizer(s) for s in pred_str]
        label_str = [normalizer(s) for s in label_str]

        # calcul du taux d'erreur
        wer = 100 * metric.compute(predictions=pred_str, references=label_str)

        return {"wer": wer}

    # Initialiser le trainer
    trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["dev"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    )
    
    # lancer l'entrainement
    trainer.train()
    
if __name__ == "__main__":
    PATH = 'kurd/'
    whisper_finetune(PATH)
    

/info/etu/m1/s2506992/miniconda3/envs/asr2526_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train examples: 3125
Dev examples: 2910
Test examples: 3230


Map (num_proc=1):  74%|███████▎  | 2299/3125 [00:38<00:13, 60.42 examples/s]


RuntimeError: One of the subprocesses has abruptly died during map operation.To debug the error, disable multiprocessing.

In [ ]:
PATH = 'kurd/'
common_voice = load_local_dataset(PATH)

In [18]:
common_voice["train"]

Dataset({
    features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__', 'audio'],
    num_rows: 0
})

In [19]:
directory = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    #chemin vers le fichier audio
PATH_TO_AUDIO = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/" + PATH + "clips/"

    # Creer les chemins dynamiques
tsv_path_train = os.path.join(directory, PATH, "train.tsv")
tsv_path_dev = os.path.join(directory, "..", "datasets", PATH, "dev.tsv")
tsv_path_test = os.path.join(directory, "..", "datasets", PATH, "test.tsv")

    # Convertir en dataframe pandas
df_train = pd.read_csv(tsv_path_train, delimiter='\t')
df_dev = pd.read_csv(tsv_path_dev, delimiter='\t')
df_test = pd.read_csv(tsv_path_test, delimiter='\t')

In [20]:
count = 0
for path in df_train['path']:
    if os.path.isfile(os.path.join(PATH_TO_AUDIO, path)):
        count += 1
print(count)

3125


In [5]:
def load_local_dataset(PATH):
    # Chemins
    base_dir = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets"
    PATH_TO_AUDIO = os.path.join(base_dir, PATH, "clips")

    # Lire les TSV
    df_train = pd.read_csv(os.path.join(base_dir, PATH, "train.tsv"), delimiter='\t')
    df_dev   = pd.read_csv(os.path.join(base_dir, PATH, "dev.tsv"), delimiter='\t')
    df_test  = pd.read_csv(os.path.join(base_dir, PATH, "test.tsv"), delimiter='\t')

    # Filtrage sans apply
    def filter_files(df):
        filtered_rows = []
        for _, row in df.iterrows():
            audio_file = os.path.join(PATH_TO_AUDIO, row['path'])
            if os.path.isfile(audio_file):
                filtered_rows.append(row)
        return pd.DataFrame(filtered_rows)

    df_train = filter_files(df_train)
    df_dev   = filter_files(df_dev)
    df_test  = filter_files(df_test)

    print("Train examples:", len(df_train))
    print("Dev examples:", len(df_dev))
    print("Test examples:", len(df_test))

    # Convertir en Dataset HuggingFace
    common_voice = DatasetDict({
        "train": Dataset.from_pandas(df_train),
        "dev": Dataset.from_pandas(df_dev),
        "test": Dataset.from_pandas(df_test),
    })

    # Ajouter colonne audio complète
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].map(
            lambda x: {"audio": os.path.join(PATH_TO_AUDIO, x["path"])},
            batched=False
        )

    # Reechantillonnage audio
    for split in ["train", "dev", "test"]:
        common_voice[split] = common_voice[split].cast_column("audio", Audio(sampling_rate=16_000))

    test_subset = common_voice["test"].select(range(2000))
    common_voice["train"] = concatenate_datasets([common_voice["train"], test_subset])

    return common_voice


In [6]:
PATH = 'kurd/'
common_voice = load_local_dataset(PATH)

Train examples: 3125
Dev examples: 2910
Test examples: 3230


Map: 100%|██████████| 3230/3230 [00:00<00:00, 11087.75 examples/s]


In [8]:
common_voice["train"]


Dataset({
    features: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment', '__index_level_0__', 'audio'],
    num_rows: 5125
})

In [7]:
common_voice = common_voice.map(prepare_dataset, fn_kwargs={"feature_extractor": feature_extractor,  "tokenizer": tokenizer}, batched=True,
    batch_size=8)

Map:   0%|          | 0/3125 [00:01<?, ? examples/s]


TypeError: list indices must be integers or slices, not str